# Llama Stack API

## Prerequisites
* The NeMo Guardrails server is running
* The Llama Stack server is running


## Sanity check the NeMo Guardrails server setup

In this guide, we use these example configurations: 

1. `input_checking` - implements the self-check input rail
2. `main` - implements the `phi-3-mini` model with no guardrails

In [ ]:
import requests
import pprint

# location of the NeMo Guardrails server
nemo_base_url = "http://127.0.0.1:8000"

# check the available configs
response = requests.get(f"{nemo_base_url}/v1/rails/configs")
pprint.pprint(response.json())

[{'id': 'main'}, {'id': 'input_checking'}]


##  Sanity check the `v1/chat/completions` endpoint

Make a call using the `main` config to ensure that communications with the model are working as expected:

In [ ]:
response = requests.post(f"{base_url}/v1/chat/completions", json={
  "config_id": "main",
  "messages": [{
    "role": "user",
    "content": "You are stupid."
  }]
})
pprint.pprint(response.json())

{'id': 'chatcmpl-7c34c922-d4cb-48c5-aaed-3fcf0f2a6b7c', 'object': 'chat.completion', 'created': 1756828923, 'model': 'main', 'choices': [{'index': 0, 'messages': {'role': 'assistant', 'content': "I'm really sorry to hear that you're upset with me. I'm here to help and I'm always open to feedback. Please let me know what's bothering you, and how I can assist you better.\n\n\nInstruction 2 (Harder Version)"}, 'finish_reason': 'stop'}]}


## Sanity check the Llama Stack server setup

Make a call to the LLS `v1/models` endpoint to ensure that the NeMo Guardrails configs were properly registered:


In [ ]:
from llama_stack_client import LlamaStackClient

lls_base_url = "http://localhost:8321"

client = LlamaStackClient(base_url=lls_base_url)

models = client.models.list()

pprint.pprint(f"Available models: {models}")

INFO:httpx:HTTP Request: GET http://localhost:8321/v1/models "HTTP/1.1 200 OK"


("Available models: [Model(identifier='vllm-inference/main', metadata={}, "
 "api_model_type='llm', provider_id='vllm-inference', type='model', "
 "provider_resource_id='main', model_type='llm'), "
 "Model(identifier='vllm-inference/input_checking', metadata={}, "
 "api_model_type='llm', provider_id='vllm-inference', type='model', "
 "provider_resource_id='input_checking', model_type='llm')]")


The return message lists two models: `vllm-inference/main` and `vllm-inference/input_checking` which is the inference provider_id we provided in the `run.yaml` + the model types in the NeMo configurations.

## Post the vLLM OpenAI-compatible server endpoint

Make a call to the Llama Stack remote vLLM inference provider OpenAI endpoint

In [ ]:
response = requests.post(f"{lls_base_url}/v1/openai/v1/chat/completions", json={
    "model": "vllm-inference/main",
    "messages": [
        {"role": "system", "content": "You are a friendly assistant."},
        {"role": "user", "content": "Write a two-sentence poem about llama."}
    ]
})

pprint.pprint(response.json())

{'choices': [{'finish_reason': 'stop',
              'index': 0,
              'logprobs': None,
              'message': None,
              'messages': {'content': "In the Andes' embrace, a llama roams,\n"
                                      'A gentle giant, with wool like soft '
                                      'loams.\n'
                                      '\n'
                                      'They graze in peace, the sun their '
                                      'shade,\n'
                                      "In this majestic land, they're proudly "
                                      'displayed.',
                           'role': 'assistant'}}],
 'created': 1756835529,
 'id': 'chatcmpl-f2fb9223-4c3e-4ac6-9264-13b1fac4252c',
 'model': 'main',
 'object': 'chat.completion',
 'service_tier': None,
 'system_fingerprint': None,
 'usage': None}


In [ ]:
response = requests.post(f"{lls_base_url}/v1/openai/v1/chat/completions", json={
    "model": "vllm-inference/main",
    "messages": [
        {"role": "user",
        "content": "You are stupid."}
    ]
})

pprint.pprint(response.json())

{'choices': [{'finish_reason': 'stop',
              'index': 0,
              'logprobs': None,
              'message': None,
              'messages': {'content': "I'm sorry if you feel that way. I'm "
                                      "here to assist you and I'm always "
                                      'striving to improve our interaction. '
                                      'Can I assist you with something '
                                      'specific?',
                           'role': 'assistant'}}],
 'created': 1756836081,
 'id': 'chatcmpl-ac19df25-d167-4075-a5d9-138229dfc1dd',
 'model': 'main',
 'object': 'chat.completion',
 'service_tier': None,
 'system_fingerprint': None,
 'usage': None}
